In [1]:
import loqs
from loqs.core import QuantumProgram
from loqs.core.recordables import MeasurementOutcomes
from loqs.backends import PyGSTiPhysicalCircuit, PyGSTiNoiseModel, QSimQuantumState
from test_codepack_5_1_3_quantinuum import Test5QCodepack

In [2]:
program = Test5QCodepack._create_program(PyGSTiPhysicalCircuit, PyGSTiNoiseModel, QSimQuantumState)
qubits = ["A0", "A1"] + [f"D{i+2}" for i in range(5)]

In [3]:
stack_ftprep_D3X_error = [
    ("Init State", None, (len(qubits),), {"qubit_labels": qubits}),
    ("Init Patch 5Q", None, ("L0", qubits)),
    # Here we inject an X error before layer 2, i.e. between CZs, on qubit 4 (0-indexed, and our qubit convention is 2 aux + 5 data)
    ("Non-FT Minus Prep + Checks", "L0", (), {"error_injections": [(2, "Gxpi", 4)]}),
    # The RUS should do a check, then see that it failed, and do it all again
    ("FT Minus Prep", "L0"),
    ("FT Logical X Measure", "L0")
]

In [4]:
program2 = QuantumProgram.from_quantum_program(program, stack_ftprep_D3X_error)

In [5]:
program2.run(1)

Program Prep minus, measure X: 100%|██████████| 1/1 [00:00<00:00,  5.45it/s]


In [6]:
program2.display()

In [9]:
loqs.tools.fttools.test_program_output(
    program2,
    collect_shot_data_args=[
        ("measurement_outcomes", 2), # Let's grab the outcomes for the first round of RUS
        ("rus_success", 3),       # And also the computed RUS success value
        ("measurement_outcomes", 5), # Let's do it again for the second round (frame 3 is the reset)
        ("rus_success", 6),       
        ("logical_measurement", -1) # And still check the logical outcome
    ],
    expected_outcomes=[
        MeasurementOutcomes({"A0": [0, 0, 1], "A1": [0, 0, 0]}),
        False,
        MeasurementOutcomes({"A0": [0, 0, 0], "A1": [0, 0, 0]}),
        True,
        1
    ],
    verbose=True
)

True